# Reproduce Table 9

This notebook reads the two saved wildcard response summaries in `script/res/`. 

NXNAME is a subset of NODATA. For NSEC, Table 9 separates NODATA with and without NXNAME. For NSEC3, Table 9 reports all NODATA together. 


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "script/res/20260406_wildcard_nxdomain_ratio.csv").is_file()), None)

if ROOT is None:
    raise FileNotFoundError("Run this notebook within the dnsnsec26 repository.")
    
RES = ROOT / "script/res"
OUT = ROOT / "notebook/results/"
ZONES = ["com", "net", "org", "umbrella", "tranco", "radar", "majestic"]
LABELS = [".com", ".net", ".org", "Umbrella", "Tranco", "Radar", "Majestic"]
COLUMNS = ["NSEC NODATA without NXNAME", "NSEC NODATA with NXNAME",
           "NSEC NXDOMAIN", "NSEC3 NODATA", "NSEC3 NXDOMAIN"]
status = pd.read_csv(RES / "20260406_wildcard_nxdomain_ratio.csv")
nxname = pd.read_csv(RES / "20260406_wildcard_nodata_nxname_ratio.csv")
nxname["type"] = nxname["type"].str.split().str[0]

# Compute each percentage using that summary's own count and total.
for frame in (status, nxname):
    assert (frame["total"] > 0).all()
    frame["fraction"] = frame["count"] / frame["total"]
    np.testing.assert_allclose(frame["fraction"], frame["perc"], atol=1e-12)


## Calculation

For each zone:

- NSEC NODATA with NXNAME = 100 × NXNAME count / NXNAME summary total.
- NSEC NODATA without NXNAME = NSEC NODATA percentage from the status summary − NSEC NXNAME percentage.
- NSEC NXDOMAIN, NSEC3 NODATA, and NSEC3 NXDOMAIN = 100 × status count / status summary total.

The subtraction uses percentages, not raw counts. NXNAME is already included in NODATA, so it is not added to the NSEC3 NODATA column. The three NSEC columns and the two NSEC3 columns each sum to 100% before rounding.


In [7]:
status_fractions = status.set_index(["zone", "type", "status"])["fraction"]
nxname_fractions = nxname.set_index(["zone", "type"])["fraction"]
rows = []
for zone in ZONES:
    nodata_nsec = status_fractions.loc[(zone, "nsec", "NODATA")]
    nxname_nsec = nxname_fractions.loc[(zone, "nsec")]
    rows.append([100 * (nodata_nsec - nxname_nsec), 100 * nxname_nsec,
                 100 * status_fractions.loc[(zone, "nsec", "NXDOMAIN")],
                 100 * status_fractions.loc[(zone, "nsec3", "NODATA")],
                 100 * status_fractions.loc[(zone, "nsec3", "NXDOMAIN")]])
table9 = pd.DataFrame(rows, index=pd.Index(LABELS, name="Dataset"), columns=COLUMNS)
np.testing.assert_allclose(table9.iloc[:, :3].sum(axis=1), 100)
np.testing.assert_allclose(table9.iloc[:, 3:].sum(axis=1), 100)
formatted = table9.map(lambda value: f"{value:.2f}%")
display(formatted)


,NSEC NODATA without NXNAME,NSEC NODATA with NXNAME,NSEC NXDOMAIN,NSEC3 NODATA,NSEC3 NXDOMAIN
Dataset,,,,,
.com,49.34%,13.41%,37.24%,55.04%,44.96%
.net,55.31%,16.87%,27.83%,60.71%,39.29%
.org,51.99%,15.98%,32.03%,54.55%,45.45%
Umbrella,63.52%,28.67%,7.81%,36.31%,63.69%
Tranco,56.43%,31.77%,11.80%,32.58%,67.42%
Radar,57.26%,31.61%,11.13%,35.35%,64.65%
Majestic,50.31%,31.43%,18.26%,30.73%,69.27%


## Paper correction and comparison

We made a mistake in the paper: the **NSEC3 NODATA and NXDOMAIN values should be swapped** for Umbrella, Tranco, Radar, and Majestic (the original value was computed via google sheet and the error was stem from copy and paste). 


## Save results

Exports are kept in `notebook/results/table9_from_summaries/` so this notebook's results are separate from earlier Table 9 notebooks. Numeric percentages retain full precision; the presentation table rounds to two decimal places.


In [8]:
table9.to_csv(OUT / "table9_percentages.csv")
formatted.to_csv(OUT / "table9_formatted.csv")
print(f"Saved results to {OUT.relative_to(ROOT)}")

Saved results to notebook/results
